# W tej części zajmujemy się załadowaniem i przetworzeniem danych do postaci użytecznej #


## 1. Konfiguracja Środowiska i Import Bibliotek ##

W celu przeprowadzenia analizy ilościowej, wykorzystujemy zestaw bibliotek dedykowanych do obliczeń naukowych, statystyki oraz wizualizacji danych finansowych.

### Stos Technologiczny:

*   **Przetwarzanie Danych:**
    *   `pandas`: Zarządzanie strukturami danych typu DataFrame, obsługa szeregów czasowych.
    *   `numpy`: Operacje wektorowe i funkcje matematyczne (np. logarytmowanie).
*   **Wizualizacja:**
    *   `matplotlib` & `seaborn`: Tworzenie technicznych wykresów cen, wolumenu oraz rozkładów statystycznych.
*   **Ekonometria i Statystyka:**
    *   `statsmodels.tsa`: Biblioteka do analizy szeregów czasowych (Testy stacjonarności ADF, analiza autokorelacji ACF/PACF).
    *   `scipy.stats`: Weryfikacja hipotez statystycznych, w tym testy normalności rozkładu (Jarque-Bera).
*   **Integracja Projektu:**
    *   `sys` & `os`: Konfiguracja ścieżek systemowych umożliwiająca importowanie autorskich modułów z katalogu `src/`.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Dodanie folderu src do ścieżki (wyjście z notebooks/ do głównego katalogu)
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_data, clean_financial_data, analyze_data_content

# Ustawienia estetyczne wykresów
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

## 2. Wczytywanie i czyszczenie danych ##

Ładowanie danych

In [3]:
from pathlib import Path

# To znajdzie główny folder projektu (BICC_2026), 
# idąc od miejsca, gdzie jest ten notebook o dwa poziomy w górę
BASE_DIR = Path(os.getcwd()).parent
path_daily = BASE_DIR / "data" / "raw" / "BICC_Daily_Markets_Anon.csv"
path_weekly = BASE_DIR / "data" / "raw" / "BICC_Weekly_Macro_Anon.csv"
path_monthly = BASE_DIR / "data" / "raw" / "BICC_Monthly_Macro_Anon.csv"


data_daily = load_data(str(path_daily))
data_weekly = load_data(str(path_weekly))
data_monthly = load_data(str(path_monthly))

✅ Ustawiono indeks czasowy na kolumnie: Date
✅ Ustawiono indeks czasowy na kolumnie: Date
✅ Ustawiono indeks czasowy na kolumnie: Date


Analiza zawartości danych dziennych

In [ ]:
print(data_daily.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1629 entries, 2020-01-01 to 2026-03-25
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 1                 1629 non-null   float64
 1   Unnamed: 2                 1629 non-null   int64  
 2   Unnamed: 3                 1628 non-null   float64
 3   Unnamed: 4                 1628 non-null   float64
 4   Unnamed: 5                 1628 non-null   float64
 5   Unnamed: 6                 1628 non-null   float64
 6   Unnamed: 7                 1628 non-null   float64
 7   Unnamed: 8                 1628 non-null   float64
 8   Unnamed: 9                 1628 non-null   float64
 9   Unnamed: 10                1628 non-null   float64
 10  Unnamed: 11                1627 non-null   float64
 11  Unnamed: 12                1627 non-null   float64
 12  Unnamed: 13                1628 non-null   float64
 13  Unnamed: 14                162

In [5]:
# zmiana nazw kolumn
daily_mapping = {
    'Unnamed: 1': 'Target_Asset_Close',
    'Unnamed: 2': 'Target_Asset_Volume',
    'Unnamed: 3': 'Equity_Index_RegionA_Close',
    'Unnamed: 4': 'Equity_Index_RegionA_Volume',
    'Unnamed: 5': 'Equity_Index_RegionB_Close',
    'Unnamed: 6': 'Equity_Index_RegionB_Volume',
    'Unnamed: 7': 'Bond_Yield_RegionA_Close',
    'Unnamed: 8': 'Bond_Yield_RegionA_Volume',
    'Unnamed: 9': 'Commodity_Energy_1_Close',
    'Unnamed: 10': 'Commodity_Energy_1_Volume',
    'Unnamed: 11': 'Safe_Haven_Metal_Close',
    'Unnamed: 12': 'Safe_Haven_Metal_Volume',
    'Unnamed: 13': 'Risk_Index_Close',
    'Unnamed: 14': 'Risk_Index_Volume'
}

# Zmiana nazw
data_daily.rename(columns=daily_mapping, inplace=True)

# Czyszczenie: wypełnianie luk w cenach ( Close ) zgodnie z instrukcją
close_cols = [col for col in data_daily.columns if '_Close' in col]

# Czyszczenie: zamiana NA w Volume na 0 (jeśli tak zdecydujecie)
volume_cols = [col for col in data_daily.columns if '_Volume' in col]

print(close_cols)
print(volume_cols)

['Target_Asset_Close', 'Equity_Index_RegionA_Close', 'Equity_Index_RegionB_Close', 'Bond_Yield_RegionA_Close', 'Commodity_Energy_1_Close', 'Safe_Haven_Metal_Close', 'Risk_Index_Close', 'Commodity_Energy_2_Close', 'Supply_Chain_Index_Close', 'ESG_Asset_Close']
['Target_Asset_Volume', 'Equity_Index_RegionA_Volume', 'Equity_Index_RegionB_Volume', 'Bond_Yield_RegionA_Volume', 'Commodity_Energy_1_Volume', 'Safe_Haven_Metal_Volume', 'Risk_Index_Volume', 'Commodity_Energy_2_Volume', 'Supply_Chain_Index_Volume', 'ESG_Asset_Volume']


In [6]:
# braki danych dla kolumn z cenami
data_daily[close_cols].isna().sum()

Target_Asset_Close            0
Equity_Index_RegionA_Close    1
Equity_Index_RegionB_Close    1
Bond_Yield_RegionA_Close      1
Commodity_Energy_1_Close      1
Safe_Haven_Metal_Close        2
Risk_Index_Close              1
Commodity_Energy_2_Close      1
Supply_Chain_Index_Close      1
ESG_Asset_Close               1
dtype: int64

In [7]:
# braki danych dla wolumenow
data_daily[volume_cols].isna().sum()

Target_Asset_Volume               0
Equity_Index_RegionA_Volume       1
Equity_Index_RegionB_Volume       1
Bond_Yield_RegionA_Volume         1
Commodity_Energy_1_Volume         1
Safe_Haven_Metal_Volume           2
Risk_Index_Volume                 1
Commodity_Energy_2_Volume      1629
Supply_Chain_Index_Volume      1629
ESG_Asset_Volume               1629
dtype: int64

In [8]:
data_daily.describe()

,Target_Asset_Close,Target_Asset_Volume,Equity_Index_RegionA_Close,Equity_Index_RegionA_Volume,Equity_Index_RegionB_Close,Equity_Index_RegionB_Volume,Bond_Yield_RegionA_Close,Bond_Yield_RegionA_Volume,Commodity_Energy_1_Close,Commodity_Energy_1_Volume,Safe_Haven_Metal_Close,Safe_Haven_Metal_Volume,Risk_Index_Close,Risk_Index_Volume,Commodity_Energy_2_Close,Commodity_Energy_2_Volume,Supply_Chain_Index_Close,Supply_Chain_Index_Volume,ESG_Asset_Close,ESG_Asset_Volume
count,1629.000000,1629.0,1628.000000,1628.0,1628.000000,1628.000000,1628.000000,1628.000000,1628.000000,1628.0,1627.000000,1.627000e+03,1628.000000,1.628000e+03,1628.000000,0.0,1628.000000,0.0,1628.000000,0.0
mean,1.114488,0.0,20.959920,0.0,2293.258535,4737.467445,73.963348,33429.675061,3.002158,0.0,4328.696863,3.141462e+07,15920.156550,5.727406e+09,49991.192875,NaN,10162.617322,NaN,6475.080467,NaN
std,0.057450,0.0,7.834496,0.0,813.141685,23519.499215,18.709723,18847.653236,1.400965,0.0,788.875401,1.527809e+07,4655.670317,2.004788e+09,45972.983045,NaN,23076.125054,NaN,2122.354283,NaN
min,0.959619,0.0,11.860000,0.0,1477.300049,0.000000,19.330000,0.000000,0.499000,0.0,2385.820068,0.000000e+00,6994.290039,2.169020e+09,3510.000000,NaN,12.000000,NaN,1612.000000,NaN
25%,1.076426,0.0,15.937500,0.0,1799.074951,98.000000,64.975002,23364.000000,1.560000,0.0,3753.560059,2.250825e+07,12321.190430,4.423320e+09,26423.750000,NaN,1385.750000,NaN,5449.250000,NaN
50%,1.102536,0.0,19.020000,0.0,1927.649963,317.500000,74.915001,30759.000000,3.684500,0.0,4224.870117,2.788970e+07,14940.169922,5.086435e+09,35191.000000,NaN,1866.000000,NaN,7018.000000,NaN
75%,1.166317,0.0,24.065000,0.0,2513.225037,865.500000,84.492498,39687.250000,4.214500,0.0,4947.514893,3.500830e+07,19490.150391,6.589732e+09,51250.000000,NaN,2484.250000,NaN,8088.000000,NaN
max,1.234111,0.0,82.690002,0.0,5318.399902,251274.000000,127.980003,235965.000000,4.988000,0.0,6173.319824,1.673299e+08,26119.849609,1.630873e+10,321415.000000,NaN,99700.000000,NaN,9747.000000,NaN


In [9]:
# usuniecie danych z pustymi wolumenami (0 lub NA)
data_daily = data_daily.drop(columns=['Target_Asset_Volume', 'Equity_Index_RegionA_Volume', 'Commodity_Energy_1_Volume', 'Commodity_Energy_2_Volume', 'Supply_Chain_Index_Volume', 'ESG_Asset_Volume'])

In [16]:
# analiza brakow danych

# od kiedy mamy komplet danych dla wszystkich kolumn
first_valid = data_daily.apply(lambda col: col.first_valid_index()).max()
print(f"Dane stają się kompletne od: {first_valid}")

# przycinamy zbior danych
data_daily = data_daily.loc[first_valid:]

Dane stają się kompletne od: 2020-01-03 00:00:00


### Stworzenie ramki danych z samymi zwrotami ###

In [17]:
# 1. Tworzymy nową ramkę na zwroty
df_returns = pd.DataFrame(index=data_daily.index)

# 2. Obliczamy Log Returns / Diffs
for col in data_daily.columns:
    if '_Close' in col:
        if 'Bond_Yield' in col:
            # Dla obligacji zmiana punktowa
            df_returns[f'{col}_diff'] = data_daily[col].diff()
        else:
            # Log Returns dla reszty
            df_returns[f'{col}_log_ret'] = np.log(data_daily[col] / data_daily[col].shift(1))

# 3. Usuwamy pierwszy wiersz, który powstał przez .diff() / .shift(1)
df_returns = df_returns.dropna()

### Analiza ramki danych ze zwrotami ###

In [19]:
df_returns.info()

df_returns.describe()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1626 entries, 2020-01-06 to 2026-03-25
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Target_Asset_Close_log_ret          1626 non-null   float64
 1   Equity_Index_RegionA_Close_log_ret  1626 non-null   float64
 2   Equity_Index_RegionB_Close_log_ret  1626 non-null   float64
 3   Bond_Yield_RegionA_Close_diff       1626 non-null   float64
 4   Commodity_Energy_1_Close_log_ret    1626 non-null   float64
 5   Safe_Haven_Metal_Close_log_ret      1626 non-null   float64
 6   Risk_Index_Close_log_ret            1626 non-null   float64
 7   Commodity_Energy_2_Close_log_ret    1626 non-null   float64
 8   Supply_Chain_Index_Close_log_ret    1626 non-null   float64
 9   ESG_Asset_Close_log_ret             1626 non-null   float64
dtypes: float64(10)
memory usage: 139.7 KB


,Target_Asset_Close_log_ret,Equity_Index_RegionA_Close_log_ret,Equity_Index_RegionB_Close_log_ret,Bond_Yield_RegionA_Close_diff,Commodity_Energy_1_Close_log_ret,Safe_Haven_Metal_Close_log_ret,Risk_Index_Close_log_ret,Commodity_Energy_2_Close_log_ret,Supply_Chain_Index_Close_log_ret,ESG_Asset_Close_log_ret
count,1626.000000,1626.000000,1626.000000,1626.000000,1626.000000,1626.000000,1626.000000,1626.000000,1626.000000,1626.000000
mean,0.000022,0.000402,0.000642,0.022073,0.000553,0.000241,0.000618,0.000844,-0.002349,0.000609
std,0.004704,0.077637,0.011328,1.867687,0.034054,0.012660,0.015695,0.054771,1.042279,0.025409
min,-0.028144,-0.442449,-0.120657,-16.840004,-0.347009,-0.132405,-0.130032,-0.352414,-4.624218,-0.177347
25%,-0.002619,-0.037858,-0.003631,-0.650000,-0.010531,-0.004047,-0.004979,-0.023252,-0.021797,-0.011065
50%,-0.000111,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.002834,0.027041,0.005818,0.850004,0.011710,0.005737,0.008116,0.023110,0.020623,0.014029
max,0.027537,0.729851,0.059054,8.620003,0.404797,0.088343,0.113528,0.412788,4.695222,0.161378


### Zdefiniowanie zmiennej docelowej - jutrzejszy zwrot ###

In [20]:
# T+1 Target
df_returns['TARGET_Next_Day_Ret'] = df_returns['Target_Asset_Close_log_ret'].shift(-1)

# Usuwamy ostatni wiersz (bo nie ma jutra)
df_returns = df_returns.dropna()

## 3. Analiza zbioru danych tygodniowych ##

### Wstępne sprawdzenie ###

In [23]:
data_weekly.head()

,Macro_Labor_RegionA,Macro_Stress_RegionA
Date,,
2020-01-04,224000,NaN
2020-01-11,207000,NaN
2020-01-18,221000,NaN
2020-01-25,210000,NaN
2020-02-01,207000,NaN


In [24]:
data_weekly.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 324 entries, 2020-01-04 to 2026-03-14
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Macro_Labor_RegionA   324 non-null    int64  
 1   Macro_Stress_RegionA  0 non-null      float64
dtypes: float64(1), int64(1)
memory usage: 7.6 KB


### Usunięcie kolumny z samymi NA i dodatkowe zmienne ###

In [32]:
data_weekly_processed = data_weekly.copy()

# dodatkowa zmienna - zmiana procentowa zatrudnienia
data_weekly_processed['Macro_Labor_RegionA_change'] = (data_weekly_processed['Macro_Labor_RegionA'] - data_weekly_processed['Macro_Labor_RegionA'].shift(1))/data_weekly_processed['Macro_Labor_RegionA'].shift(1)

# usunięcie NA 
data_weekly_processed.drop(columns = ['Macro_Stress_RegionA'], inplace = True)

data_weekly_processed.dropna(inplace = True)
data_weekly_processed.head()

,Macro_Labor_RegionA,Macro_Labor_RegionA_change
Date,,
2020-01-11,207000,-0.075893
2020-01-18,221000,0.067633
2020-01-25,210000,-0.049774
2020-02-01,207000,-0.014286
2020-02-08,202000,-0.024155


In [34]:
data_weekly_processed.describe()

,Macro_Labor_RegionA,Macro_Labor_RegionA_change
count,3.230000e+02,323.000000
mean,4.430929e+05,0.025110
std,6.874287e+05,0.544824
min,1.900000e+05,-0.206616
25%,2.140000e+05,-0.038904
50%,2.250000e+05,-0.008757
75%,3.015000e+05,0.019422
max,6.137000e+06,9.673993


## 4. Złączenie danych dziennych i tygodniowych ##

In [35]:
# 1. Sortujemy indeksy (wymóg dla merge_asof)
df_returns = df_returns.sort_index()
data_weekly_processed = data_weekly_processed.sort_index()

# 2. łączenie
# 'backward' sprawi, że w poniedziałek 13.01 model będzie widział dane z soboty 11.01,
# ale nie dowie się nic o danych z przyszłej soboty 18.01.
df_combined = pd.merge_asof(
    df_returns, 
    data_weekly_processed[['Macro_Labor_RegionA_change']], 
    left_index=True, 
    right_index=True, 
    direction='backward'
)

# 3. Wypełnienie luk (ffill)
# Dane tygodniowe "wskakują" raz na tydzień, przez resztę dni musimy je powtarzać.
df_combined['Macro_Labor_RegionA_change'] = df_combined['Macro_Labor_RegionA_change'].ffill()

In [37]:
df_combined.head(15)

,Target_Asset_Close_log_ret,Equity_Index_RegionA_Close_log_ret,Equity_Index_RegionB_Close_log_ret,Bond_Yield_RegionA_Close_diff,Commodity_Energy_1_Close_log_ret,Safe_Haven_Metal_Close_log_ret,Risk_Index_Close_log_ret,Commodity_Energy_2_Close_log_ret,Supply_Chain_Index_Close_log_ret,ESG_Asset_Close_log_ret,TARGET_Next_Day_Ret,Macro_Labor_RegionA_change
Date,,,,,,,,,,,,
2020-01-06,-0.000849,-0.012200,0.010914,0.310005,0.012782,-0.005541,0.006192,-0.055823,-0.071990,-0.030257,0.003223,NaN
2020-01-07,0.003223,-0.004342,0.003569,-0.640007,0.008796,0.001792,-0.000234,-0.028916,-0.064855,0.012852,-0.003870,NaN
2020-01-08,-0.003870,-0.024965,-0.009204,-2.829994,0.025400,0.003534,0.007424,0.002929,-0.023019,-0.018527,-0.003730,NaN
2020-01-09,-0.003730,-0.070056,-0.003667,-0.070000,-0.008574,0.006162,0.008631,0.017810,-0.001294,0.024098,-0.000189,NaN
2020-01-10,-0.000189,0.001594,0.003731,-0.389999,-0.017921,-0.001677,-0.002561,-0.019901,0.002587,-0.018827,0.000500,NaN
2020-01-13,0.000500,-0.019293,-0.005860,-0.780006,0.012524,-0.002600,0.011533,0.009585,-0.011696,-0.002835,0.001713,-0.075893
2020-01-14,0.001713,0.005666,-0.003882,0.290001,-0.016367,-0.001271,-0.004113,-0.064234,-0.002618,-0.011830,-0.000434,-0.075893
2020-01-15,-0.000434,0.002418,0.006269,-0.489998,-0.016639,-0.001569,0.000249,-0.011119,0.006532,0.027122,0.002074,-0.075893
2020-01-16,0.002074,-0.008084,-0.001999,0.620003,0.011677,0.001373,0.009838,-0.005831,0.000000,0.013093,-0.001294,-0.075893


## 5. Analiza zbioru danych miesiecznych ##

In [42]:
monthly_data = data_monthly.copy()
monthly_data.head()

,Macro_Inflation_RegionA,Macro_Production_RegionA,Macro_Production_RegionB,Macro_Global_Uncertainty
Date,,,,
2020-01-01,259.127,101.0338,100.191651,191.721791
2020-02-01,259.250,101.3735,NaN,196.493257
2020-03-01,258.076,97.4078,NaN,318.464859
2020-04-01,256.032,84.5619,84.792934,328.132639
2020-05-01,255.802,85.9604,NaN,415.743351


In [39]:
monthly_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 74 entries, 2020-01-01 to 2026-02-01
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Macro_Inflation_RegionA   73 non-null     float64
 1   Macro_Production_RegionA  74 non-null     float64
 2   Macro_Production_RegionB  15 non-null     float64
 3   Macro_Global_Uncertainty  71 non-null     float64
dtypes: float64(4)
memory usage: 2.9 KB


### Usunięcie kolumny Macro_Production_RegionB i załatwienie pozostałych NA ###

In [43]:
monthly_data.drop(columns = ['Macro_Production_RegionB'], inplace = True)

monthly_data = monthly_data.ffill()

print(monthly_data.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 74 entries, 2020-01-01 to 2026-02-01
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Macro_Inflation_RegionA   74 non-null     float64
 1   Macro_Production_RegionA  74 non-null     float64
 2   Macro_Global_Uncertainty  74 non-null     float64
dtypes: float64(3)
memory usage: 2.3 KB
None


### Dodanie nowych kolumn - zmian w czasie ###

In [45]:
monthly_data['Macro_Inflation_RegionA_change'] = monthly_data['Macro_Inflation_RegionA'].pct_change()
monthly_data['Macro_Production_RegionA_change'] = monthly_data['Macro_Production_RegionA'].diff()
monthly_data['Macro_Global_Uncertainty_change'] = monthly_data['Macro_Global_Uncertainty'].diff()

print(monthly_data.head())


            Macro_Inflation_RegionA  Macro_Production_RegionA  \
Date                                                            
2020-01-01                  259.127                  101.0338   
2020-02-01                  259.250                  101.3735   
2020-03-01                  258.076                   97.4078   
2020-04-01                  256.032                   84.5619   
2020-05-01                  255.802                   85.9604   

            Macro_Global_Uncertainty  Macro_Inflation_RegionA_change  \
Date                                                                   
2020-01-01                191.721791                             NaN   
2020-02-01                196.493257                        0.000475   
2020-03-01                318.464859                       -0.004528   
2020-04-01                328.132639                       -0.007920   
2020-05-01                415.743351                       -0.000898   

            Macro_Production_RegionA_ch

### Usunięcie pierwszych miesięcy - mamy NA ###

In [46]:
monthly_data.dropna(inplace = True)

## 5. Połączenie danych miesięcznych z połączoną resztą danych ##

In [49]:
# Łączymy bazę (Daily + Weekly) z nowymi danymi Monthly
df_final = pd.merge_asof(
    df_combined.sort_index(), 
    monthly_data.sort_index(), 
    left_index=True, 
    right_index=True, 
    direction='backward' # Model widzi ostatni raport miesięczny
)

df_final.tail()

,Target_Asset_Close_log_ret,Equity_Index_RegionA_Close_log_ret,Equity_Index_RegionB_Close_log_ret,Bond_Yield_RegionA_Close_diff,Commodity_Energy_1_Close_log_ret,Safe_Haven_Metal_Close_log_ret,Risk_Index_Close_log_ret,Commodity_Energy_2_Close_log_ret,Supply_Chain_Index_Close_log_ret,ESG_Asset_Close_log_ret,TARGET_Next_Day_Ret,Macro_Labor_RegionA_change,Macro_Inflation_RegionA,Macro_Production_RegionA,Macro_Global_Uncertainty,Macro_Inflation_RegionA_change,Macro_Production_RegionA_change,Macro_Global_Uncertainty_change
Date,,,,,,,,,,,,,,,,,,
2026-03-23,-0.001504,-0.023806,-0.037065,-12.250000,-0.013066,0.01319,0.012080,-0.044376,-0.009284,0.023402,0.004054,-0.037559,327.46,102.551,371.095352,0.00267,0.1547,0.0
2026-03-24,0.004054,0.030134,-0.001091,4.549995,0.013294,0.00125,-0.007725,-0.047731,-0.023846,0.027435,0.000000,-0.037559,327.46,102.551,371.095352,0.00267,0.1547,0.0
2026-03-25,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,-0.053027,0.000000,-0.023052,-0.002388,-0.037559,327.46,102.551,371.095352,0.00267,0.1547,0.0
2026-03-25,-0.002388,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.037559,327.46,102.551,371.095352,0.00267,0.1547,0.0
2026-03-25,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.037559,327.46,102.551,371.095352,0.00267,0.1547,0.0


### Usunięcie NA z finalnego zbioru danych ###

In [52]:
df_final.dropna(inplace = True)
print(df_final.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1605 entries, 2020-02-03 to 2026-03-25
Data columns (total 18 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Target_Asset_Close_log_ret          1605 non-null   float64
 1   Equity_Index_RegionA_Close_log_ret  1605 non-null   float64
 2   Equity_Index_RegionB_Close_log_ret  1605 non-null   float64
 3   Bond_Yield_RegionA_Close_diff       1605 non-null   float64
 4   Commodity_Energy_1_Close_log_ret    1605 non-null   float64
 5   Safe_Haven_Metal_Close_log_ret      1605 non-null   float64
 6   Risk_Index_Close_log_ret            1605 non-null   float64
 7   Commodity_Energy_2_Close_log_ret    1605 non-null   float64
 8   Supply_Chain_Index_Close_log_ret    1605 non-null   float64
 9   ESG_Asset_Close_log_ret             1605 non-null   float64
 10  TARGET_Next_Day_Ret                 1605 non-null   float64
 11  Macro_Labor_RegionA_chang

### Macierz korelacji dla danych finalnych ###

In [55]:
# 3. Macierz korelacji - co rusza Twoim assetem?
corrs = df_final.corr()['TARGET_Next_Day_Ret'].sort_values(ascending=False)
print("Korelacja z jutrzejszym zwrotem Targetu:")
print(corrs)

Korelacja z jutrzejszym zwrotem Targetu:
TARGET_Next_Day_Ret                   1.000000
Risk_Index_Close_log_ret              0.117808
Equity_Index_RegionB_Close_log_ret    0.114835
Safe_Haven_Metal_Close_log_ret        0.105454
Macro_Labor_RegionA_change            0.081793
ESG_Asset_Close_log_ret               0.061229
Macro_Global_Uncertainty              0.059975
Supply_Chain_Index_Close_log_ret      0.050693
Macro_Production_RegionA_change       0.021592
Target_Asset_Close_log_ret            0.019199
Macro_Inflation_RegionA               0.004565
Macro_Global_Uncertainty_change      -0.000836
Macro_Production_RegionA             -0.020238
Macro_Inflation_RegionA_change       -0.029002
Bond_Yield_RegionA_Close_diff        -0.030423
Commodity_Energy_2_Close_log_ret     -0.062791
Commodity_Energy_1_Close_log_ret     -0.074378
Equity_Index_RegionA_Close_log_ret   -0.091746
Name: TARGET_Next_Day_Ret, dtype: float64


## 6. Zapisanie plikow csv do folderu data/processed ##

In [ ]:


# 1. ścieżka do folderu
output_dir = '../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Utworzono folder: {output_dir}")

# 2. Słownik ramek do zapisania
dfs_to_save = {
    'data_daily_cleaned.csv': data_daily,       # Surowe, ale oczyszczone z NaN i ffill
    'df_returns_daily.csv': df_returns,         # Same zwroty dzienne + TARGET
    'data_weekly_macro.csv': data_weekly,       # Oryginalne dane tygodniowe
    'data_monthly_macro.csv': monthly_data,     # Oryginalne dane miesięczne
    'df_final_master.csv': df_final             # Zintegrowany zbiór (Daily + Weekly + Monthly)
}

# 3. Pętla zapisująca pliki
for filename, df in dfs_to_save.items():
    path = os.path.join(output_dir, filename)
    df.to_csv(path)
    print(f"Zapisano: {path} | Wymiary: {df.shape}")

print("\n--- Wszystkie pliki są gotowe ---")

Zapisano: ../data/processed\data_daily_cleaned.csv | Wymiary: (1627, 14)
Zapisano: ../data/processed\df_returns_daily.csv | Wymiary: (1625, 11)
Zapisano: ../data/processed\data_weekly_macro.csv | Wymiary: (324, 2)
Zapisano: ../data/processed\data_monthly_macro.csv | Wymiary: (73, 6)
Zapisano: ../data/processed\df_final_master.csv | Wymiary: (1605, 18)

--- Wszystkie pliki są gotowe do wysłania na GitHub! ---
